In [ ]:
# hide
import mido
import numpy as np
import pyquist as pq

In [ ]:
NOTE_OFF, NOTE_ON = 0x80, 0x90  # the status types, in the top four bits of the status byte


def note_on(channel: int, pitch: int, velocity: int) -> bytes:
    return bytes([NOTE_ON | channel, pitch, velocity])


def note_off(channel: int, pitch: int, velocity: int = 0) -> bytes:
    return bytes([NOTE_OFF | channel, pitch, velocity])


# Twinkle, Twinkle, Little Star at 120 BPM, as real-time messages: (time, message)
pitches = [60, 60, 67, 67, 69, 69, 67]
durations = [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 1.0]
onsets = np.cumsum([0.0] + durations[:-1])
messages = []
for onset, duration, pitch in zip(onsets, durations, pitches):
    messages.append((onset, note_on(0, pitch, 100)))
    messages.append((onset + duration, note_off(0, pitch)))
messages.sort(key=lambda m: m[0])  # note offs and note ons interleave in time

for time, message in messages[:6]:
    print(f"{time:4.1f} s: {message.hex(' ')}")
print(f"{len(messages)} messages, {sum(len(m) for _, m in messages)} bytes")

In [ ]:
# Store the same messages in a standard MIDI file, with times in ticks
TICKS_PER_BEAT = 480
track = mido.MidiTrack()
previous = 0
for time, message in messages:
    tick = round(time * 2 * TICKS_PER_BEAT)       # 120 BPM: 2 beats per second
    msg = mido.Message.from_bytes(message)
    track.append(msg.copy(time=tick - previous))  # SMF times are deltas between messages
    previous = tick
midi_file = mido.MidiFile(ticks_per_beat=TICKS_PER_BEAT, tracks=[track])
midi_file.save("twinkle.mid")
with open("twinkle.mid", "rb") as f:
    print(f"twinkle.mid is {len(f.read())} bytes")

# Read it back as a real-time score of note on / note off events...
score, metronome = pq.Score.from_midi("twinkle.mid", as_notes=False)
for event in score[:4]:
    print(event)

# ...or as notes with durations, which we can render as usual
def sine(pitch: int, duration: float, **kwargs) -> pq.Audio:
    t = np.arange(int(duration * 44100)) / 44100
    x = np.sin(2 * np.pi * pq.helper.pitch_to_frequency(pitch) * t) * np.exp(-4 * t)
    return pq.Audio(x, sample_rate=44100)

notes, metronome = pq.Score.from_midi("twinkle.mid")
pq.play(notes.render(sine, metronome=metronome))